# Bronze Layer — Dynamic Auto Loader Streaming Ingestion

## Architecture
This notebook reads the `framework.pipeline_config` table and **automatically provisions
and runs one Auto Loader stream per active source**. No code changes are needed when a new
source is added — just insert a row into `pipeline_config`.

## Auto Loader Design
| Feature | Setting |
|---|---|
| Format detection | `file_format` column in config |
| Schema inference | `cloudFiles.inferColumnTypes = true` |
| Schema evolution | `cloudFiles.schemaHints` + `mergeSchema = true` |
| Schema persistence | `cloudFiles.schemaLocation` per source |
| File notifications | `cloudFiles.useNotifications = true` (Event Grid + Queue) |
| Incremental trigger | `trigger(availableNow=True)` — drains backlog then stops |
| Checkpointing | Per-source checkpoint under `{CHECKPOINT_BASE}bronze/{source_name}/` |
| Audit columns | `ingestion_date`, `load_timestamp`, `source_file` auto-added |
| Partitioning | `partition_col` from config (default: `ingestion_date`) |
| CDF | `delta.enableChangeDataFeed = true` on every Bronze table |

> **Supported formats:** JSON, CSV, Parquet — driven entirely by `file_format` in config.

In [ ]:
%run ./fw_0.config

In [ ]:
# ── Step 1: Ensure Bronze schema exists ──────────────────────────────────────

spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}
    MANAGED LOCATION '{BRONZE_PATH}'
    COMMENT 'Raw ingestion layer — Auto Loader Delta tables'
""")
print(f"Bronze schema ready: {CATALOG_NAME}.{BRONZE_SCHEMA}")

In [ ]:
# ── Step 2: Core Auto Loader ingestion function ───────────────────────────────

from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException

def ingest_bronze(cfg) -> dict:
    """
    Run an Auto Loader stream for a single pipeline_config row.
    Returns a result dict with status and rows_affected.
    """
    source_name   = cfg.source_name
    source_path   = f"{BRONZE_PATH}{cfg.source_path}"
    file_format   = cfg.file_format.lower()
    target_table  = fq(BRONZE_SCHEMA, cfg.bronze_table)
    partition_col = cfg.partition_col or "ingestion_date"
    zorder_cols   = [c.strip() for c in (cfg.zorder_cols or "").split(",") if c.strip()]
    checkpoint    = f"{CHECKPOINT_BASE}bronze/{source_name}/"
    schema_loc    = f"{SCHEMA_BASE}bronze/{source_name}/"

    print(f"\n[BRONZE] Starting: {source_name}  →  {target_table}")
    print(f"         Source   : {source_path} ({file_format})")
    print(f"         Checkpoint: {checkpoint}")

    # ── Build readStream ────────────────────────────────────────────────────
    reader = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format",          file_format)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation",   schema_loc)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        # Event Grid notifications — eliminates directory-listing overhead at scale
        .option("cloudFiles.useNotifications", "true")
    )

    # Format-specific options
    if file_format == "json":
        reader = reader.option("multiLine", "true")
    elif file_format == "csv":
        reader = (
            reader
            .option("header",    str(cfg.csv_header).lower() if cfg.csv_header is not None else "true")
            .option("delimiter", cfg.csv_delimiter or ",")
            .option("inferSchema", "true")
        )
    # Parquet: no additional options needed — schema is embedded in file

    stream_df = (
        reader
        .load(source_path)
        .withColumn("ingestion_date",  F.current_date())
        .withColumn("load_timestamp",  F.current_timestamp())
        .withColumn("source_file",     F.col("_metadata.file_path"))
    )

    # ── Build writeStream ───────────────────────────────────────────────────
    writer = (
        stream_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint)
        .option("mergeSchema",        "true")
        .trigger(availableNow=True)
    )

    # Use toTable() — Unity Catalog managed Delta table, honours TBLPROPERTIES if table exists
    query = writer.toTable(target_table)
    query.awaitTermination()

    # ── Post-ingest: set TBLPROPERTIES if this is a new table ──────────────
    spark.sql(f"""
        ALTER TABLE {target_table}
        SET TBLPROPERTIES (
            'delta.enableChangeDataFeed' = 'true',
            'quality'                    = 'bronze'
        )
    """)

    # ── Post-ingest: OPTIMIZE + ZORDER ─────────────────────────────────────
    if zorder_cols:
        spark.sql(f"OPTIMIZE {target_table} ZORDER BY ({', '.join(zorder_cols)})")
        print(f"         OPTIMIZE ZORDER BY ({', '.join(zorder_cols)}) complete.")
    else:
        spark.sql(f"OPTIMIZE {target_table}")
        print(f"         OPTIMIZE complete.")

    row_count = spark.table(target_table).count()
    print(f"[BRONZE] Done: {target_table}  →  {row_count:,} total rows")
    return {"status": "SUCCESS", "rows": row_count}

In [ ]:
# ── Step 3: Iterate configs and ingest all active sources ─────────────────────
# Each source is processed sequentially so checkpoint directories don't conflict.
# For true parallelism, split into separate Databricks Workflow tasks per config_id.

configs  = get_pipeline_configs(filter_active=True)
results  = []

print(f"Active configs: {len(configs)}")

for cfg in configs:
    try:
        result = ingest_bronze(cfg)
        log_pipeline_run(cfg.config_id, "bronze", "SUCCESS", result["rows"])
        results.append({"source": cfg.source_name, **result})

    except Exception as e:
        error_msg = str(e)
        print(f"[BRONZE] FAILED: {cfg.source_name}  →  {error_msg}")
        log_pipeline_run(cfg.config_id, "bronze", "FAILURE", error_msg=error_msg)
        results.append({"source": cfg.source_name, "status": "FAILURE", "error": error_msg})

# ── Summary ──────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("BRONZE INGESTION SUMMARY")
print("="*60)
for r in results:
    status = r.get('status', 'UNKNOWN')
    rows   = r.get('rows', 'N/A')
    print(f"  {r['source']:20s}  {status:10s}  {str(rows):>12s} rows")

In [ ]:
# ── Step 4: Data Quality Gate — fail the notebook if any source failed ────────
# Databricks Workflow will mark the task as FAILED and trigger retry/alert.

failures = [r for r in results if r.get("status") == "FAILURE"]
if failures:
    failed_sources = ", ".join(r["source"] for r in failures)
    raise RuntimeError(
        f"[DQ FAIL] Bronze ingestion failed for: {failed_sources}. "
        "Check pipeline_run_log for details."
    )

print("[DQ PASS] All Bronze sources ingested successfully.")